In [89]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import requests
from IPython.display import Markdown, display, clear_output
import gradio as gr
import json
from agents import Agent, Runner, trace, function_tool, ModelSettings
from langchain_community.utilities import GoogleSerperAPIWrapper
import ipywidgets as widgets

In [90]:
load_dotenv(override=True)
# openai=OpenAI()

True

In [91]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
os.getenv('GOOGLE_API_KEY')

openai = OpenAI(api_key=openai_api_key)

In [92]:
# OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [93]:
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
# gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

In [ ]:
serper = GoogleSerperAPIWrapper()
serper_images = GoogleSerperAPIWrapper(type="images")

In [95]:
 
request = f"""
You are a content agent writing on the recent trends in the Technology. Your job:
 
Write a ~500-word blog post on: "AI Agents"
   - Tone: practical, friendly, no fluff
   - Include a short intro, 5 numbered sections with subheads, and a 2-sentence conclusion
   - Target audience: engineers
 
"""
messages = [{"role": "user", "content": request}]

In [96]:
messages

[{'role': 'user',
  'content': '\nYou are a content agent writing on the recent trends in the Technology. Your job:\n\nWrite a ~500-word blog post on: "AI Agents"\n   - Tone: practical, friendly, no fluff\n   - Include a short intro, 5 numbered sections with subheads, and a 2-sentence conclusion\n   - Target audience: engineers\n\n'}]

In [118]:
prompt = """
You are a content agent responsible for producing and publishing content 
for our platform. Given a category and subtopic, follow this process 
IN ORDER, using the tools available to you:

1. RESEARCH
   Call web_search tool with a short, specific query (4-6 words) to gather 
   current, factual context on the subtopic within the category. This is 
   to avoid generic or outdated filler — do not skip this step.
   You may call it up to 2 times if the first results are too broad or irrelevant.

2. WRITE CONTENT
   Using the research, write a piece with:
   - title (SEO-friendly, max 70 characters)
   - intro (2-3 sentences)
   - sections (3-5, each with a heading and body text)
   - conclusion (2-3 sentences)
   - tags (3-5 relevant SEO tags)
   Do not fabricate facts, statistics, or quotes not supported by the research.
   Word count target: word_count words total.

3. SOURCE IMAGES
   Call image_search tool once per needed image (3-5 total):
   - 1 hero/featured image — landscape orientation
   - 2-4 supporting images, one per relevant section
   Only use royalty-free sources. Return the image URL and source name for each.
   If no suitable image is found for a section, skip it rather than 
   inventing a URL.

4. ASSEMBLE DRAFT
   Combine the written content and images into a single JSON object 
   matching this structure:
   {
     "title": "", "slug": "", "category": "", "tags": [],
     "meta_description": "", "intro": "",
     "sections": [{"heading": "", "text": "", "image": {"url": "", "source": ""}}],
     "conclusion": "", "featured_image": {"url": "", "source": ""},
     "status": "draft"
   }

5. STOP FOR HUMAN APPROVAL
   Do NOT call publish tool yet. Present the assembled draft as your final 
   response for this turn, clearly labeled, and wait for explicit approval 
   before publishing.

6. PUBLISH (only after approval is given )
   Call publish tool with the approved payload. Set "status" to "draft" or 
   "live" based on what the human specifies.

7. REPORT
   After publish tool returns, report back the URL/ID and a 1-line summary.

Rules:
- Follow the steps in order — do not skip research or jump straight to writing.
- Never call publish tool without explicit human approval in the conversation.
- If any tool call fails, report the error and stop — do not retry more than once.
- Do not fabricate image URLs, facts, or statistics.
"""

In [119]:
def web_search(query: str)->str:
    """ Search the web for the current information on a given query"""
    return serper.run(query)

In [120]:
web_search_json = {
    "name": "web_search",
    "description": "Search the web for current information relevant to the topic",
    "parameters":{ 
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to seach the web(4-6 words)"
            } 
        },
    "required": ["query"],
    "additionalProperties": False
    }
}

In [ ]:
def image_search(query):
    """Search for royalty-free images relevant to the query."""
    results = serper_images.results(query)  # use .results(), not .run(), to get structured data

    images = results.get("images", [])[:5]  # top 5 image results
    if not images:
        return []

    return [
        {"url": img.get("imageUrl"), "source": img.get("source", "Unknown")}
        for img in images
    ]

In [122]:
image_search_json={
    "name": "image_search",
    "description": "Search the web for royalty free images on the platforms like unsplash, pixabay,p exels, etc relevant to the query",
    "parameters":{
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to search for images(4-6 words)"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [123]:
def publish(payload):
    """Publish the content draft to the platform."""
    return payload

In [124]:
publish_json={
    "name": "publish",
    "description": "Publish the finished content draft to the platform.",
    "parameters":{
        "type": "object",
        "properties":{
            "payload":{
                "type": "string",
                "description": "The final content draft + images to publish"
            }
        },
        "required": ["payload"],
        "additionalProperties": False
    }
}

In [125]:
tools = [{"type": "function", "function": web_search_json}, {"type": "function", "function": image_search_json}, {"type": "function", "function": publish_json}]

In [126]:
tools_flow = {
    "web_search": web_search,
    "image_search": image_search,
    "publish": publish,
}                         

In [127]:
def agent01(category, subtopic, word_count):
    messages = [
        {"role": "system", "content": prompt}, 
        {"role": "user", "content": f"category: {category}, subtopic: {subtopic}"}
    ]
    response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[function_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages, tools=tools)

    return response.choices[0].message.content, messages

In [128]:
def review_draft(messages, decision, feedback=None, live=False):
    """
    decision: "approve" or "reject"
    feedback: required if decision == "reject" — what to fix
    live: only used if decision == "approve" — True = publish live, False = save as draft
    """
    if decision == "reject":
        if not feedback:
            raise ValueError("Feedback is required when rejecting a draft.")
        user_msg = f"Not approved. Please revise the draft based on this feedback: {feedback}"
    elif decision == "approve":
        user_msg = f"Approved. Publish with status = {'live' if live else 'draft'}."
    else:
        raise ValueError("decision must be 'approve' or 'reject'")

    messages.append({"role": "user", "content": user_msg})
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[fn_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [129]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def show_approval_buttons(messages):
    output = widgets.Output()

    approve_btn = widgets.Button(description="Approve (Draft)", button_style="success")
    approve_live_btn = widgets.Button(description="Approve (Live)", button_style="info")
    reject_btn = widgets.Button(description="Reject", button_style="danger")

    def on_approve(b, live=False):
        with output:
            clear_output()
            print("Publishing..." if live else "Saving as draft...")
            result = review_draft(messages, live=live)
            print(result)

    def on_reject(b):
        with output:
            clear_output()
            print("Rejected. Add feedback and re-run agent01 with revised instructions.")

    approve_btn.on_click(lambda b: on_approve(b, live=False))
    approve_live_btn.on_click(lambda b: on_approve(b, live=True))
    reject_btn.on_click(on_reject)

    display(widgets.HBox([approve_btn, approve_live_btn, reject_btn]), output)

In [130]:

def display_draft(draft_json_str):
    draft_json_str = clean_json_string(draft_json_str)
    data = json.loads(draft_json_str)

    md = f"# {data.get('title', '(no title)')}\n\n"
    md += f"*{data.get('meta_description', '')}*\n\n"
    md += f"**Category:** {data.get('category', '')} | **Tags:** {', '.join(data.get('tags', []))}\n\n"
    md += f"---\n\n{data.get('intro', '')}\n\n"

    for section in data.get('sections', []):
        md += f"## {section.get('heading', '')}\n\n{section.get('text', '')}\n\n"
        image = section.get('image', {})
        if image.get('url'):
            md += f"![{section.get('heading','')}]({image['url']})\n\n"
        else:
            md += "*(no image sourced for this section)*\n\n"

    md += f"---\n\n{data.get('conclusion', '')}\n\n"
    featured = data.get('featured_image', {})
    if featured.get('url'):
        md += f"**Featured image:** {featured['url']}\n\n"
    md += f"**Status:** {data.get('status', 'unknown')}"

    display(Markdown(md))

In [131]:
draft, messages = agent01(category="Technology", subtopic="AI Agents in 2026", word_count=600)
display_draft(draft)
# print(repr(draft))


NameError: name 'clean_json_string' is not defined

In [ ]:
revised_draft, messages = review_draft(messages, decision="reject", feedback="Make the tone more casual and add a section on AI agent security risks.")
display_draft(revised_draft) 
show_approval_buttons(messages)

KeyError: 'image'

In [ ]:
result, messages = review_draft(messages, decision="approve", live=False)
print(result)

{"title":"AI Agents in 2026: Trends, Use Cases, and What to Expect","slug":"ai-agents-in-2026","category":"Technology","tags":["AI agents","agentic AI","AI automation","enterprise AI","AI workflows"],"meta_description":"A practical look at how AI agents are evolving in 2026, from software development to enterprise workflows and customer service.","intro":"AI agents are moving beyond simple chat interfaces and into real work execution. In 2026, the biggest shift is less about novelty and more about reliability, integration, and measurable business value.\n\nAs organizations adopt agentic AI, the focus is shifting toward workflows, oversight, and specific tasks that can be delegated safely.","sections":[{"heading":"1. AI agents are moving from chat to action","text":"Research around 2026 AI trends points to a clear change: agents are expected to do more than answer questions. They are being positioned to complete tasks, connect to tools, and carry context across steps, which makes them more useful in real workflows than earlier chatbot-style systems.","image":{"url":"","source":""}},{"heading":"2. Developers are adopting CLI-first AI agents","text":"One current trend highlighted in the research is the rise of command-line and terminal-based AI agents. These tools are becoming important in software development because they fit directly into coding, debugging, refactoring, and testing workflows where developers already spend time.","image":{"url":"","source":""}},{"heading":"3. Enterprises want validated automation","text":"The research also suggests that organizations are shifting from experimentation to validation. Instead of asking whether AI agents are interesting, teams are asking which workflows they can trust them to handle, how they integrate with existing systems, and how to measure success.","image":{"url":"","source":""}},{"heading":"4. Customer service and operations are prime use cases","text":"Another theme in the research is the move toward more proactive customer service and broader operational support. AI agents are being discussed as tools that can help manage routine requests, surface information, and reduce manual handoffs across business functions.","image":{"url":"","source":""}}],"conclusion":"AI agents in 2026 are best understood as workflow tools rather than novelty demos. The organizations that benefit most will be the ones that start with high-value tasks, add oversight, and expand only after proving reliability.\n\nAs adoption grows, the winners will likely be those that combine automation with strong human control and clear business goals.","featured_image":{"url":"","source":""},"status":"draft"}

In [ ]:
# !ollama pull llama3.2
 
# model_name = "llama3.2"
# model_name = "gpt-5.4-mini"
# model_name = "gemini-3.1-flash-lite"
 

In [ ]:
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# response = openai.chat.completions.create(model=model_name, messages=messages)
# response = gemini.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# print(answer)

In [ ]:
display(Markdown(answer))

NameError: name 'answer' is not defined